# Part 2 · Notebook 02 — Calculus and optimization

**Sessions:** S2 (Calculus & optimization) · [Lesson plan](../../docs/lessons/PART_02_QUANT_TOOLKIT.md)

**You will:**
1. Approximate option P&L with delta and gamma, and see where it fails.
2. Solve the minimum-variance portfolio in closed form.
3. Add constraints with `cvxpy`.
4. Find an implied volatility with Newton's method.

How these notebooks work: the loading and plotting code is written for you. Cells marked **✍️ Your turn** need 1–5 lines from you.
If your answer does not match yet, the notebook continues with the reference answer so nothing else breaks.

In [ ]:
import sys
from pathlib import Path
for d in (Path.cwd(), Path.cwd().parent):       # p2lib.py is in notebooks/part02/
    sys.path.insert(0, str(d))
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
import p2lib as p

p.use_course_style()
pd.set_option("display.float_format", "{:,.4f}".format)
prices = p.load_prices()          # dates × 10 tickers (course data via P2_DATA, else synthetic)
rets = p.log_returns(prices)      # daily log returns
prices.tail(3)

## 1. Taylor approximation: delta and gamma

$\Delta V \approx \Delta \cdot \Delta S + \tfrac{1}{2} \Gamma \cdot \Delta S^2$

In [ ]:
S, K, T, r, q, vol = 100.0, 100.0, 0.25, 0.04, 0.0, 0.20      # ✏️ change me
base = p.bsm_call(S, K, T, r, q, vol)
print({k: round(v, 4) for k, v in base.items()})

✍️ **Your turn** — replace each `...` and run the cell. `p.check` tells you if you are right.

In [ ]:
moves = np.array([1.0, 5.0, 15.0])
# ✍️ delta-gamma approximation of the change in call value for each move (use base["delta"], base["gamma"])
approx = ...
approx = p.check("delta-gamma approximation", approx, p.delta_gamma_pnl(base["delta"], base["gamma"], moves))

In [ ]:
dS = np.linspace(-25, 25, 201)
full = np.array([p.bsm_call(S + x, K, T, r, q, vol)["price"] for x in dS]) - base["price"]
fig, ax = plt.subplots()
ax.plot(dS, full, label="Full repricing")
ax.plot(dS, base["delta"] * dS, label="Delta only", ls="--")
ax.plot(dS, p.delta_gamma_pnl(base["delta"], base["gamma"], dS), label="Delta + gamma", ls=":")
ax.set(title="Change in call value vs move in the underlying", xlabel="ΔS"); ax.legend(); plt.show()
pd.DataFrame({"move": moves, "full": [p.bsm_call(S + m, K, T, r, q, vol)["price"] - base["price"] for m in moves],
              "delta-gamma": approx})

## 2. Minimum-variance portfolio (Lagrange multipliers)

$w = \dfrac{\Sigma^{-1}\mathbf{1}}{\mathbf{1}^\top \Sigma^{-1} \mathbf{1}}$ — compute it with `np.linalg.solve`, never by inverting the matrix.

✍️ **Your turn** — replace each `...` and run the cell. `p.check` tells you if you are right.

In [ ]:
cov_ann = rets.cov().to_numpy() * 252
# ✍️ x = solve(Σ, 1), then normalize so the weights sum to 1
w_mv = ...
w_mv = p.check("minimum-variance weights", w_mv, p.min_variance_weights(cov_ann))

## 3. Adding constraints with `cvxpy` (no short selling)

In [ ]:
import cvxpy as cp
x = cp.Variable(len(cov_ann))
cp.Problem(cp.Minimize(cp.quad_form(x, cp.psd_wrap(cov_ann))), [cp.sum(x) == 1, x >= 0]).solve()
weights = pd.DataFrame({"closed form (may short)": w_mv, "long-only (cvxpy)": x.value}, index=rets.columns)
weights["long-only (cvxpy)"] = weights["long-only (cvxpy)"].clip(lower=0).round(4)
vols = {c: np.sqrt(weights[c] @ cov_ann @ weights[c]) for c in weights}
print({k: f"{v:.2%}" for k, v in vols.items()})
weights

In [ ]:
# Efficient frontier (long-only): minimum variance for each target return
mu = rets.mean().to_numpy() * 252
targets = np.linspace(mu.min(), mu.max(), 25)
frontier = []
for m in targets:
    x = cp.Variable(len(mu))
    prob = cp.Problem(cp.Minimize(cp.quad_form(x, cp.psd_wrap(cov_ann))), [cp.sum(x) == 1, x >= 0, mu @ x == m])
    prob.solve()
    if x.value is not None:
        frontier.append((np.sqrt(prob.value), m))
f = np.array(frontier)
fig, ax = plt.subplots()
ax.plot(f[:, 0] * 100, f[:, 1] * 100, label="Efficient frontier (long-only)")
ax.scatter(np.sqrt(np.diag(cov_ann)) * 100, mu * 100, color="#52514e", s=20, zorder=3, label="Single assets")
for i, t in enumerate(rets.columns):
    ax.annotate(t, (np.sqrt(cov_ann[i, i]) * 100, mu[i] * 100), fontsize=8, color="#52514e", xytext=(3, 3), textcoords="offset points")
ax.set(xlabel="volatility (%)", ylabel="mean log return (%/yr)", title="Efficient frontier"); ax.legend(); plt.show()

## 4. Root finding: implied volatility with Newton's method

Update: $\sigma \leftarrow \sigma - \dfrac{C(\sigma) - C_{market}}{\text{vega}(\sigma)}$

✍️ **Your turn** — replace each `...` and run the cell. `p.check` tells you if you are right.

In [ ]:
market_price, sigma = 5.0, 0.30                 # ✏️ market call price; starting guess

def newton_step(sigma):
    c = p.bsm_call(S, K, T, r, q, sigma)
    # ✍️ return the next sigma (use c["price"], c["vega"], market_price)
    return ...

for _ in range(20):
    nxt = newton_step(sigma)
    if nxt is Ellipsis:
        break                                        # not done yet
    sigma = nxt
from scipy.optimize import brentq
sigma = p.check("implied volatility", sigma,
                brentq(lambda v: p.bsm_call(S, K, T, r, q, v)["price"] - market_price, 1e-4, 5))
print(f"Implied volatility: {sigma:.4%}")

## Questions
1. For which size of move is the delta-gamma approximation good enough? What does that mean for risk reports during crashes?
2. Why do the closed-form weights include short positions, and why does the long-only answer have higher volatility?
3. When can Newton's method fail for implied volatility? (Hint: vega near zero.)